# Preprocessing — Jalur **IndoBERT**

**Tujuan.** Menyiapkan teks untuk *fine-tuning* **IndoBERT**. Notebook membaca data
berlabel dari **MongoDB Atlas** (`raw_comments`, `in_balanced_set = true`), menerapkan
pembersihan **minimal**, lalu menulis ke koleksi **`processed_bert`**.

**Mengapa minimal?** Berbeda dengan TF-IDF, IndoBERT sudah dilatih pada bahasa
Indonesia alami dan memiliki *tokenizer subword* sendiri yang memahami konteks. Maka
**stemming, penghapusan stopword, dan normalisasi slang TIDAK dilakukan** — imbuhan,
kata fungsi, tanda baca, dan negasi justru penting bagi model. Cukup satu tahap:

- **`clean_for_bert`** — huruf kecil; buang URL, mention, dan emoji. **Tanda baca dan
  angka dipertahankan.**

> **Konsistensi pembanding.** Prosedur split identik dengan notebook SVM (urut
> `comment_id` + `seed = 42`, dilakukan sebelum preprocessing) sehingga test/val
> kedua model sama persis.

## 0. Dependency

Pasang driver MongoDB & scikit-learn (untuk split). Pembersihan IndoBERT hanya butuh `re` bawaan Python.

In [1]:
%pip install -q "pymongo[srv]" dnspython certifi python-dotenv scikit-learn pandas numpy

Note: you may need to restart the kernel to use updated packages.


## 1. Baca dataset berlabel dari MongoDB

Mengambil subset seimbang 1.000/kelas (3.000 dokumen, `in_balanced_set = true`) dari `raw_comments`. Kredensial dari `.env` (lokal) atau `getpass` (Colab).

In [2]:
import os, pandas as pd
from pymongo import MongoClient
import certifi

MONGO_URI = os.environ.get("MONGO_URI", "")
if not MONGO_URI:
    try:
        from dotenv import load_dotenv; load_dotenv(); MONGO_URI = os.environ.get("MONGO_URI","")
    except Exception: pass
if not MONGO_URI:
    from getpass import getpass; MONGO_URI = getpass("MONGO_URI: ")
DB_NAME = os.environ.get("MONGO_DB_NAME","youtube_sentiment")
client = MongoClient(MONGO_URI, tlsCAFile=certifi.where(), serverSelectionTimeoutMS=20000)
client.admin.command("ping"); print("Koneksi MongoDB OK.")
LABELS = ["Negatif","Netral","Positif"]
docs = list(client[DB_NAME]["raw_comments"].find({"in_balanced_set":True},
            {"_id":0,"comment_id":1,"video_id":1,"text":1,"label":1}))
df = pd.DataFrame(docs); df = df[df["label"].isin(LABELS)].copy()
print(f"{len(df)} dok balanced dari Mongo"); print(df["label"].value_counts().reindex(LABELS).to_string())

Koneksi MongoDB OK.


3000 dok balanced dari Mongo
label
Negatif    1000
Netral     1000
Positif    1000


## 2. Fungsi preprocessing (inline)

Hanya `clean_for_bert` (pembersihan minimal) — ditanam langsung agar notebook *self-contained*. Tidak ada stemming/stopword/slang, sehingga morfologi dan negasi tetap utuh untuk dipahami tokenizer IndoBERT.

In [3]:
import re

_RE_URL = re.compile('https?://\\S+|www\\.\\S+', re.IGNORECASE)
_RE_MENTION = re.compile('@\\w+')
_RE_EMOJI = re.compile('[😀-🙏🌀-🗿🚀-\U0001f6ff🜀-🝿🞀-\U0001f7ff🠀-\U0001f8ff🤀-🧿🨀-\U0001fa6f🩰-\U0001faff✂-➰Ⓜ-🉑🤦-🤷\u200d♀-♂☀-⭕⏏⏩⌚️〰]+', re.UNICODE)
_RE_MULTI_SPACE = re.compile('\\s+')

def clean_for_bert(text: str) -> str:
    """Cleaning minimal untuk jalur IndoBERT (morfologi terjaga)."""
    if not text or not isinstance(text, str):
        return ""
    t = text.lower()
    t = _RE_URL.sub(" ", t)
    t = _RE_MENTION.sub(" ", t)
    t = _RE_EMOJI.sub(" ", t)
    t = re.sub(r"[^\w\s.,!?;:'\"()-]", " ", t)
    t = _RE_MULTI_SPACE.sub(" ", t)
    return t.strip()


def make_text_bert(text: str) -> str:
    return clean_for_bert(text or "")

## 3. Transformasi (satu tahap)

Memperlihatkan bahwa teks nyaris tidak berubah — tanda baca, imbuhan, dan negasi tetap dipertahankan; hanya URL/emoji yang dibersihkan dan huruf dikecilkan.

In [4]:
def trace_bert(t):
    print(f"RAW              : {t!r}")
    print(f"clean_for_bert   : {clean_for_bert(t)!r}   <- FINAL (morfologi & tanda baca utuh)")
trace_bert("Ijazahnya tidak palsu kok, jgn asal tuduh tanpa bukti")

RAW              : 'Ijazahnya tidak palsu kok, jgn asal tuduh tanpa bukti'
clean_for_bert   : 'ijazahnya tidak palsu kok, jgn asal tuduh tanpa bukti'   <- FINAL (morfologi & tanda baca utuh)


### Tabel *before → after* (sampel acak)

Membandingkan teks asli dengan hasil jalur IndoBERT (`bert`). Perbedaannya halus — memang disengaja.

In [5]:
import pandas as pd
pd.set_option("display.max_colwidth", 55)
_s=df.sample(6,random_state=7).copy(); _s["bert"]=_s["text"].astype(str).map(make_text_bert)
_s[["label","text","bert"]]

,label,text,bert
1306,Negatif,Keren cadas RH,keren cadas rh
2037,Negatif,Orang2 aneh.kata2 maaf aja di ributin d permasalahk...,orang2 aneh.kata2 maaf aja di ributin d permasalahk...
568,Negatif,Kandangin seumur hidup... biar Indonesia ngak gaduh...,kandangin seumur hidup... biar indonesia ngak gaduh...
1897,Positif,Kalau nonton mas Roy Suryo aku senyum2 sendiri.\nSe...,kalau nonton mas roy suryo aku senyum2 sendiri. sem...
2498,Negatif,Bu Diana mantap. IQ nya tinggi sekali.,bu diana mantap. iq nya tinggi sekali.
1111,Positif,Inget yang asli yang ada badaknya 🦏🦏🦏🦏🦏🦏🦏🦏🦏🦏🦏🦏🦏🦏🦏🦏,inget yang asli yang ada badaknya


## 4. Terapkan + split identik (70/20/10)

Prosedur split sama persis dengan notebook SVM (urut `comment_id`, `seed = 42`, sebelum preprocessing) → test/val identik. Lalu terapkan `make_text_bert` per split.

In [6]:
from sklearn.model_selection import train_test_split
LABEL2ID={l:i for i,l in enumerate(LABELS)}
df = df.sort_values("comment_id").reset_index(drop=True)
df["label_id"]=df["label"].map(LABEL2ID)
tmp, te = train_test_split(df, test_size=0.10, stratify=df["label_id"], random_state=42)
tr, va = train_test_split(tmp, test_size=0.20/0.90, stratify=tmp["label_id"], random_state=42)
splits={}
for nm,part in [("train",tr),("val",va),("test",te)]:
    part=part.copy(); part["bert"]=part["text"].astype(str).map(make_text_bert)
    b=len(part); part=part[part["bert"].str.len()>0]
    if b-len(part): print(f"[{nm}] {b-len(part)} dibuang (kosong)")
    part["split"]=nm
    splits[nm]=part.reset_index(drop=True)
for nm,part in splits.items():
    print(f"[{nm}] n={len(part)} | "+" ".join(f"{k}={v}" for k,v in part['label'].value_counts().reindex(LABELS).items()))

[train] n=2100 | Negatif=700 Netral=700 Positif=700
[val] n=600 | Negatif=200 Netral=200 Positif=200
[test] n=300 | Negatif=100 Netral=100 Positif=100


## 5. EDA & ringkasan kualitas

Distribusi kelas, statistik panjang teks, ukuran kosakata, baris kosong/duplikat, dan pemeriksaan kebocoran antar split (harus `0`).

In [7]:
# === EDA & ringkasan kualitas ===
import numpy as np
proc = pd.concat(splits.values(), ignore_index=True)
print("Distribusi kelas (after preprocessing):")
print(proc["label"].value_counts().reindex(LABELS).to_string())

wl_o = proc["text"].astype(str).str.split().map(len)
wl_p = proc["bert"].astype(str).str.split().map(len)
print(f"\nPanjang (kata)  ORIG : mean {wl_o.mean():.1f} | median {wl_o.median():.0f} | p95 {wl_o.quantile(.95):.0f} | max {wl_o.max()}")
print(f"Panjang (kata)  BERT : mean {wl_p.mean():.1f} | median {wl_p.median():.0f} | p95 {wl_p.quantile(.95):.0f} | max {wl_p.max()}")

vocab = set(t for s in proc["bert"].astype(str) for t in s.split())
dup = proc["bert"].duplicated().sum()
empty = (wl_p<=1).sum()
cross = (proc.groupby("bert")["split"].nunique()>1).sum()
print(f"\nVocab unik       : {len(vocab)}")
print(f"Baris <=1 kata    : {empty} ({empty/len(proc)*100:.1f}%)")
print(f"Duplikat teks     : {dup} ({dup/len(proc)*100:.1f}%)")
print(f"Teks lintas split (LEAKAGE): {cross}  -> harus 0")

Distribusi kelas (after preprocessing):
label
Negatif    1000
Netral     1000
Positif    1000

Panjang (kata)  ORIG : mean 15.6 | median 11 | p95 44 | max 224
Panjang (kata)  BERT : mean 15.5 | median 11 | p95 44 | max 224

Vocab unik       : 10972
Baris <=1 kata    : 2 (0.1%)
Duplikat teks     : 2 (0.1%)
Teks lintas split (LEAKAGE): 0  -> harus 0


## 6. Term diskriminatif per kelas

Kata paling khas per kelas via skor *lift* (proporsi-dalam-kelas ÷ proporsi-global). Catatan: teks BERT masih mengandung stopword, tetapi skor *lift* otomatis menekan kata yang tersebar merata di semua kelas, sehingga kata bermakna tetap menonjol.

In [8]:
# === Top term diskriminatif per kelas (over-representasi vs global) ===
from collections import Counter
cnt = {l: Counter() for l in LABELS}; tot = Counter(); nclass = {}
for l in LABELS:
    sub = proc[proc["label"]==l]; nclass[l]=len(sub)
    for s in sub["bert"].astype(str):
        for t in set(s.split()): cnt[l][t]+=1; tot[t]+=1
N=len(proc)
def top(l,k=12,minc=5):
    out=[]
    for t,c in cnt[l].items():
        if tot[t]<minc: continue
        lift=(c/nclass[l])/(tot[t]/N)   # >1 = lebih sering di kelas ini drpd rata2
        out.append((t,round(lift,2),c))
    return sorted(out,key=lambda x:-x[1])[:k]
for l in LABELS:
    print(f"[{l}] " + ", ".join(f"{t}({lift})" for t,lift,c in top(l)))

[Negatif] pembenci(3.0), panci.(3.0), panik(3.0), mulutnya(3.0), dikit(3.0), daripada(3.0), stress(3.0), fitnah(2.74), panci(2.73), tahan(2.7), kebencian(2.67), tompel(2.65)
[Netral] putusan(3.0), keputusan(3.0), host(3.0), kpk(3.0), harta(3.0), vs(2.67), bingung(2.62), tidur(2.62), kedua(2.57), kemajuan(2.57), bencana(2.45), istilah(2.4)
[Positif] kampus(3.0), poto(3.0), terbongkar(3.0), kejanggalan(3.0), dian(3.0), kepalsuan(3.0), membongkar(3.0), ova(3.0), 5(2.62), tunjukin(2.6), perlihatkan(2.57), podcast(2.57)


## 7. Tulis hasil ke koleksi `processed_bert`

Menyimpan tiap dokumen (kolom `bert`, `label`, `label_id`, `split`) via *upsert* berdasarkan `comment_id` — idempotent.

In [9]:
from pymongo import UpdateOne
OUT=client[DB_NAME]["processed_bert"]; ops=[]
for nm,part in splits.items():
    for r in part.to_dict("records"):
        ops.append(UpdateOne({"comment_id":r["comment_id"]},{"$set":{
            "comment_id":r["comment_id"],"video_id":r.get("video_id"),"text":r["text"],
            "bert":r["bert"],"label":r["label"],"label_id":int(r["label_id"]),"split":nm}},upsert=True))
OUT.bulk_write(ops,ordered=False)
print("processed_bert:",OUT.count_documents({}),"dok")

processed_bert: 3000 dok


✅ **Selesai.** Koleksi `processed_bert` berisi teks siap fine-tune beserta penanda `split`. **Langkah berikutnya:** `indobert_finetune_colab.ipynb` (Colab/GPU).